# Introduction to Vector Stores


- Use Langchain documentation to create vector stores using chromadb, milvus, weaviate, pinecone: https://python.langchain.com/docs/integrations/vectorstores/
- Create embeddings of a document and index into vector stores: https://huggingface.co/blog/getting-started-with-embeddings
- For the task you will be indexing the book: https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf
- Experiment different vector stores and other variables for and present your findings for optimizing retrieval (the most appropriate parts should be returned for any query)
- Hint: chunking

**Steps**
- Go through the given readings. Take an hour at max.
- Index document as vectors. Use any documentation to help you, but avoid using AI Tools.
- For embedding model, use an open source model from huggingface.
- Query should return most appropriate parts in relevance to the query. *Remember, your task is to build effective retrieval*


**Resources for understanding vector search**
- https://weaviate.io/blog/vector-search-explained


**Additional resources for understanding embeddings**
- https://cohere.com/llmu/text-embeddings
- https://docs.cohere.com/v2/docs/embeddings
- https://docs.cohere.com/v2/docs/playground-overview

In [12]:
!pip install langchain-chroma

In [16]:
!pip install chromadb langchain-chroma langchain-huggingface sentence-transformers

In [17]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

In [18]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:
# Your documents
documents = ["Document 1 text", "Document 2 text", "Document 3 text"]

# Create vector store with embeddings
vector_store = Chroma.from_texts(
    texts=documents,
    embedding=embeddings,
    persist_directory="./my_vector_store"
)

In [20]:
results = vector_store.similarity_search("Your query here")

In [23]:
!pip install PyPDF2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 7.5 MB/s eta 0:00:00


In [25]:
# 1. Download and read the PDF
import requests
from PyPDF2 import PdfReader
from io import BytesIO

url = "https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf"
response = requests.get(url)
reader = PdfReader(BytesIO(response.content))
full_text = "".join([page.extract_text() for page in reader.pages])

In [29]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [30]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

strategies = [
    {"chunk_size": 200, "overlap": 20},
    {"chunk_size": 500, "overlap": 50},
    {"chunk_size": 1000, "overlap": 100},
]

chunks_by_strategy = {}

for s in strategies:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=s["chunk_size"],
        chunk_overlap=s["overlap"],
        separators=["\n\n", "\n", ". ", " ", ""]
    )

    chunks_by_strategy[f"{s['chunk_size']}_{s['overlap']}"] = splitter.split_text(full_text)

In [31]:
# Example metadata creation for a chunk
chunk_metadata = {
    "source": "crime_and_punishment.pdf",
    "chunk_id": 0,
    "chapter": "CHAPTER I", # Extract from text if possible
    "contains_dialogue": True, # Heuristic check for quotes
    "character_mentioned": "Raskolnikov", # Simple keyword check
}

In [34]:
!pip install chromadb langchain-chroma langchain-huggingface sentence-transformers PyPDF2 requests

In [36]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
import requests
from PyPDF2 import PdfReader
from io import BytesIO

In [37]:
# Download Crime and Punishment PDF
url = "https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf"
response = requests.get(url)
reader = PdfReader(BytesIO(response.content))

# Extract all text
full_text = ""
for page in reader.pages:
    full_text += page.extract_text()

print(f"Total characters: {len(full_text)}")
print(f"First 500 characters:\n{full_text[:500]}")

Total characters: 1172015
First 500 characters:
Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.Crime and Punishment
By Fyodor Dostoevsky
Crime and Punishment Translator’s Preface
A few words about Dostoevsky himself may help the Eng -
lish reader to understand his work.
Dostoevsky was the son of a doctor. His parents were 
very hard- working and deeply religious people, but so poor 
that they lived with their five children in only two rooms. 
The father a


In [38]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,        # Medium chunks
    chunk_overlap=50,      # 50 character overlap
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_text(full_text)
print(f"Total chunks created: {len(chunks)}")
print(f"Sample chunk:\n{chunks[0][:200]}...")

Total chunks created: 2519
Sample chunk:
Download free eBooks of classic literature, books and 
novels at Planet eBook. Subscribe to our free eBooks blog 
and email newsletter.Crime and Punishment
By Fyodor Dostoevsky
Crime and Punishment T...


In [39]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embeddings model loaded successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings model loaded successfully!


In [40]:
# Index all chunks into Chroma
vector_store = Chroma.from_texts(
    texts=chunks,
    embedding=embeddings,
    persist_directory="./crime_punishment_chroma"  # Saves locally
)

print(f"Successfully indexed {len(chunks)} chunks into Chroma!")

Successfully indexed 2519 chunks into Chroma!


In [41]:
query = "What was the name of Raskolnikov's landlady?"
results = vector_store.similarity_search(query, k=3)

print(f"Query: {query}\n")
print("Top 3 Results:")
print("-" * 50)
for i, doc in enumerate(results, 1):
    print(f"{i}. {doc.page_content[:300]}...")
    print("-" * 30)

Query: What was the name of Raskolnikov's landlady?

Top 3 Results:
--------------------------------------------------
1. complete stranger, who was looking at him very inquisi -
tively. He was a young man with a beard, wearing a full, 
short- waisted coat, and looked like a messenger. The land -
lady was peeping in at the half-opened door. Raskolnikov 
sat up.
‘Who is this, Nastasya?’ he asked, pointing to the young 
...
------------------------------
2. She felt for some reason ashamed and uneasy.
*****
On the way to Porfiry’s, Razumihin was obviously ex -
cited.
‘That’s capital, brother,’ he repeated several times, ‘and I 
am glad! I am glad!’
‘What are you glad about?’ Raskolnikov thought to him -
self.
‘I didn’t know that you pledged things at t...
------------------------------
3. was wide open on the stairs; he could hear exclamations and 
discussion. Razumihin’s room was fairly large; the company 
consisted of fifteen people. Raskolnikov stopped in the en -
try, where two of th

In [44]:
# Add metadata to chunks first
from langchain_core.documents import Document

docs_with_metadata = []

for i, chunk in enumerate(chunks):
    doc = Document(
        page_content=chunk,
        metadata={
            "chunk_id": i,
            "source": "crime_and_punishment.pdf",
            "contains_raskolnikov": "Raskolnikov" in chunk,
            "contains_marmeladov": "Marmeladov" in chunk,
        }
    )
    docs_with_metadata.append(doc)

# Recreate vector store with metadata
vector_store_with_meta = Chroma.from_documents(
    documents=docs_with_metadata,
    embedding=embeddings,
    persist_directory="./crime_punishment_chroma_meta"
)

# Search only chunks containing "Raskolnikov"
filtered_results = vector_store_with_meta.similarity_search(
    query="What did Raskolnikov do?",
    k=3,
    filter={"contains_raskolnikov": True}
)

print("Filtered Results (only chunks mentioning Raskolnikov):")

for doc in filtered_results:
    print(f"Chunk {doc.metadata['chunk_id']}: {doc.page_content[:200]}...")
    print("-" * 30)

Filtered Results (only chunks mentioning Raskolnikov):
Chunk 999: was greatly relieved by Pulcheria Alexandrovna’s questions, 
which showered in a continual stream upon him.
He talked for three quarters of an hour, being constantly 0 Free eBooks at Planet eBook.co...
------------------------------
Chunk 2228: gave him a certain original, even a mysterious character. 
As concerned his sister, Raskolnikov was convinced that 
Svidrigaïlov would not leave her in peace. But it was too 
tiresome and unbearable t...
------------------------------
Chunk 2350: Raskolnikov walked with lagging steps, as though still hesi -
tating whether to go or not. But nothing would have turned 
him back: his decision was taken.
‘Besides, it doesn’t matter, they still know...
------------------------------


In [47]:
print("Vector store saved to ./crime_punishment_chroma")


Vector store saved to ./crime_punishment_chroma


In [49]:
# Test small chunks
splitter_small = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20,  # Corrected: use chunk_overlap, not overlap
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks_small = splitter_small.split_text(full_text)

vector_store_small = Chroma.from_texts(
    texts=chunks_small,
    embedding=embeddings,
    persist_directory="./chroma_small"
)

# Test large chunks
splitter_large = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,  # Corrected: use chunk_overlap, not overlap
    separators=["\n\n", "\n", ". ", " ", ""]
)
chunks_large = splitter_large.split_text(full_text)

vector_store_large = Chroma.from_texts(
    texts=chunks_large,
    embedding=embeddings,
    persist_directory="./chroma_large"
)

print("All three Chroma stores created!")
print(f"Small chunks: {len(chunks_small)}")
print(f"Medium chunks: {len(chunks)}")
print(f"Large chunks: {len(chunks_large)}")

All three Chroma stores created!
Small chunks: 6837
Medium chunks: 2519
Large chunks: 1292


In [50]:
!pip install chromadb langchain-chroma langchain-huggingface sentence-transformers PyPDF2 requests

In [52]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import requests
from PyPDF2 import PdfReader
from io import BytesIO

url = "https://www.planetebook.com/free-ebooks/crime-and-punishment.pdf"
response = requests.get(url)
reader = PdfReader(BytesIO(response.content))

full_text = ""
for page in reader.pages:
    full_text += page.extract_text()

print(f"Loaded {len(full_text)} characters")

Loaded 1172015 characters


In [53]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)
print("Embeddings model ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings model ready!


In [54]:
# Test configurations
configs = [
    {"chunk_size": 200, "overlap": 20, "name": "Small"},
    {"chunk_size": 500, "overlap": 50, "name": "Medium"},
    {"chunk_size": 1000, "overlap": 100, "name": "Large"},
]

stores = {}
for config in configs:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=config["chunk_size"],
        chunk_overlap=config["overlap"],
        separators=["\n\n", "\n", ". ", " ", ""]
    )
    chunks = splitter.split_text(full_text)

    store = Chroma.from_texts(
        texts=chunks,
        embedding=embeddings,
        persist_directory=f"./chroma_{config['name']}"
    )
    stores[config["name"]] = {
        "store": store,
        "chunks": chunks,
        "size": config["chunk_size"],
        "overlap": config["overlap"],
        "count": len(chunks)
    }
    print(f"{config['name']}: {len(chunks)} chunks")

print("\nAll stores created!")

Small: 6837 chunks
Medium: 2519 chunks
Large: 1292 chunks

All stores created!


In [55]:
def add_metadata(chunks):
    documents = []
    for i, chunk in enumerate(chunks):
        # Character detection
        has_raskolnikov = "Raskolnikov" in chunk
        has_marmeladov = "Marmeladov" in chunk
        has_sonia = "Sonia" in chunk

        # Chapter detection (simple)
        chapter = "Unknown"
        if "CHAPTER I" in chunk:
            chapter = "1"
        elif "CHAPTER II" in chunk:
            chapter = "2"
        elif "CHAPTER III" in chunk:
            chapter = "3"

        doc = Document(
            page_content=chunk,
            metadata={
                "chunk_id": i,
                "chapter": chapter,
                "has_raskolnikov": has_raskolnikov,
                "has_marmeladov": has_marmeladov,
                "has_sonia": has_sonia,
                "length": len(chunk)
            }
        )
        documents.append(doc)
    return documents

# Create metadata-rich documents for Medium store
medium_chunks = stores["Medium"]["chunks"]
docs_with_meta = add_metadata(medium_chunks)

# Recreate Medium store with metadata
vector_store = Chroma.from_documents(
    documents=docs_with_meta,
    embedding=embeddings,
    persist_directory="./chroma_medium_with_meta"
)
print(f"Indexed {len(docs_with_meta)} chunks with metadata!")

Indexed 2519 chunks with metadata!


In [56]:
test_queries = [
    "Who is Raskolnikov and what does he do?",
    "Describe Marmeladov's confession in the tavern.",
    "What happened when Raskolnikov visited the pawnbroker?",
    "Why did Raskolnikov faint?",
    "What is the theme of suffering in the novel?",
]

for query in test_queries[:3]:  # Test first 3
    results = vector_store.similarity_search(query, k=3)
    print(f"\nQuery: {query}")
    print("-" * 50)
    for i, doc in enumerate(results, 1):
        print(f"{i}. Chapter {doc.metadata['chapter']}:")
        print(f"   {doc.page_content[:150]}...")
        print(f"   (Contains Raskolnikov: {doc.metadata['has_raskolnikov']})")
    print("=" * 50)


Query: Who is Raskolnikov and what does he do?
--------------------------------------------------
1. Chapter Unknown:
   was greatly relieved by Pulcheria Alexandrovna’s questions, 
which showered in a continual stream upon him.
He talked for three quarters of an hour, b...
   (Contains Raskolnikov: True)
2. Chapter Unknown:
   Raskolnikov listened attentively.
‘That was five weeks ago, sir. Yes…. As soon as Kateri -
na Ivanovna and Sonia heard of it, mercy on us, it was as 
...
   (Contains Raskolnikov: True)
3. Chapter Unknown:
   vitch. He was obviously in an exceedingly good humour 
and perhaps a trifle exhilarated. ‘If it’s on business you are 
rather early.[*] It’s only a ch...
   (Contains Raskolnikov: True)

Query: Describe Marmeladov's confession in the tavern.
--------------------------------------------------
1. Chapter Unknown:
   Hamlet’ were heard in the entry. The room was filled with Crime and Punishment noise. The tavern-keeper and the boys were busy with the 
new-c

In [57]:
def filtered_search(query, character_filter=None, k=5):
    """Search only chunks mentioning a specific character."""
    filter_dict = {}
    if character_filter == "Raskolnikov":
        filter_dict = {"has_raskolnikov": True}
    elif character_filter == "Marmeladov":
        filter_dict = {"has_marmeladov": True}
    elif character_filter == "Sonia":
        filter_dict = {"has_sonia": True}

    if filter_dict:
        results = vector_store.similarity_search(query, k=k, filter=filter_dict)
    else:
        results = vector_store.similarity_search(query, k=k)

    return results

# Test filtered search
query = "What did Raskolnikov do?"
print(f"Query: {query}")
print("\nWithout filter:")
results = filtered_search(query)
for doc in results[:2]:
    print(f"- {doc.page_content[:100]}...")

print("\nWith Raskolnikov filter:")
filtered = filtered_search(query, character_filter="Raskolnikov")
for doc in filtered[:2]:
    print(f"- {doc.page_content[:100]}...")

Query: What did Raskolnikov do?

Without filter:
- was greatly relieved by Pulcheria Alexandrovna’s questions, 
which showered in a continual stream up...
- gave him a certain original, even a mysterious character. 
As concerned his sister, Raskolnikov was ...

With Raskolnikov filter:
- was greatly relieved by Pulcheria Alexandrovna’s questions, 
which showered in a continual stream up...
- gave him a certain original, even a mysterious character. 
As concerned his sister, Raskolnikov was ...


In [58]:
def evaluate_retrieval(query, results):
    """Score results based on word overlap with query."""
    print(f"\nQuery: {query}")
    print("-" * 60)

    query_words = set(query.lower().split())

    for i, doc in enumerate(results, 1):
        chunk_words = set(doc.page_content.lower().split())
        # Calculate word overlap
        common_words = query_words.intersection(chunk_words)
        word_score = len(common_words) / len(query_words) if query_words else 0

        print(f"{i}. Relevance Score: {word_score:.2f}")
        print(f"   Chapter: {doc.metadata['chapter']}")
        print(f"   Preview: {doc.page_content[:150]}...")
        print(f"   Contains: ", end="")
        if doc.metadata['has_raskolnikov']: print("Raskolnikov ", end="")
        if doc.metadata['has_marmeladov']: print("Marmeladov ", end="")
        if doc.metadata['has_sonia']: print("Sonia", end="")
        print("\n" + "-" * 40)

# Evaluate on test queries
for query in test_queries[:3]:
    results = vector_store.similarity_search(query, k=5)
    evaluate_retrieval(query, results)


Query: Who is Raskolnikov and what does he do?
------------------------------------------------------------
1. Relevance Score: 0.25
   Chapter: Unknown
   Preview: was greatly relieved by Pulcheria Alexandrovna’s questions, 
which showered in a continual stream upon him.
He talked for three quarters of an hour, b...
   Contains: Raskolnikov 
----------------------------------------
2. Relevance Score: 0.50
   Chapter: Unknown
   Preview: Raskolnikov listened attentively.
‘That was five weeks ago, sir. Yes…. As soon as Kateri -
na Ivanovna and Sonia heard of it, mercy on us, it was as 
...
   Contains: Raskolnikov Sonia
----------------------------------------
3. Relevance Score: 0.62
   Chapter: Unknown
   Preview: vitch. He was obviously in an exceedingly good humour 
and perhaps a trifle exhilarated. ‘If it’s on business you are 
rather early.[*] It’s only a ch...
   Contains: Raskolnikov 
----------------------------------------
4. Relevance Score: 0.38
   Chapter: Unknown
   Prev

In [59]:
def test_all_configs(query):
    """Test the same query across all chunk sizes."""
    results = {}

    for name, data in stores.items():
        store = data["store"]
        chunks = data["chunks"]

        # Search
        search_results = store.similarity_search(query, k=3)

        # Calculate average relevance
        query_words = set(query.lower().split())
        scores = []
        for doc in search_results:
            chunk_words = set(doc.page_content.lower().split())
            common = query_words.intersection(chunk_words)
            scores.append(len(common) / len(query_words) if query_words else 0)

        avg_score = sum(scores) / len(scores) if scores else 0
        results[name] = {
            "avg_score": avg_score,
            "chunk_count": len(chunks),
            "sample": search_results[0].page_content[:100] if search_results else "No results"
        }

    return results

# Test a query across all configurations
query = "Who is Raskolnikov?"
comparison = test_all_configs(query)

print(f"Query: {query}\n")
print("Configuration Comparison:")
print("-" * 60)
for name, data in comparison.items():
    print(f"{name} Chunks:")
    print(f"  - Avg Relevance: {data['avg_score']:.3f}")
    print(f"  - Total chunks: {data['chunk_count']}")
    print(f"  - Sample: {data['sample']}...")
    print("-" * 40)

Query: Who is Raskolnikov?

Configuration Comparison:
------------------------------------------------------------
Small Chunks:
  - Avg Relevance: 0.000
  - Total chunks: 6837
  - Sample: hin; but Raskolnikov did not and perhaps could not answer....
----------------------------------------
Medium Chunks:
  - Avg Relevance: 0.111
  - Total chunks: 2519
  - Sample: was greatly relieved by Pulcheria Alexandrovna’s questions, 
which showered in a continual stream up...
----------------------------------------
Large Chunks:
  - Avg Relevance: 0.111
  - Total chunks: 1292
  - Sample: Raskolnikov started.
‘Svidrigaïlov! Svidrigaïlov has shot himself!’ he cried.
‘What, do you know Svi...
----------------------------------------


In [60]:
# Find best performing configuration
best = None
best_score = 0

for name, data in comparison.items():
    if data['avg_score'] > best_score:
        best_score = data['avg_score']
        best = name

print("=" * 60)
print("BEST CONFIGURATION RECOMMENDATION")
print("=" * 60)
print(f"Chunk Size: {stores[best]['size']} characters")
print(f"Overlap: {stores[best]['overlap']} characters")
print(f"Total Chunks: {stores[best]['count']}")
print(f"Average Relevance Score: {best_score:.3f}")
print("\nThis configuration provides the best balance of:")
print("- Context preservation (enough text)")
print("- Precision (not too much filler)")
print("- Retrieval speed (manageable number of chunks)")
print("=" * 60)

BEST CONFIGURATION RECOMMENDATION
Chunk Size: 500 characters
Overlap: 50 characters
Total Chunks: 2519
Average Relevance Score: 0.111

This configuration provides the best balance of:
- Context preservation (enough text)
- Precision (not too much filler)
- Retrieval speed (manageable number of chunks)
